In [ ]:
import pandas as pd
import numpy as np

# Load your data
df = pd.read_csv('pl_stats.csv')

# 1. Standardize columns: Remove spaces or special chars if needed
df.columns = df.columns.str.replace(' ', '_')

# 2. Handle Missing Data: 
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

# 3. Feature Normalization:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df[numeric_cols])
df_scaled = pd.DataFrame(df_scaled, columns=numeric_cols)

Vis 1: Shooting Volume vs. Accuracy

Caption: This scatter plot segments players by threat level. Elite threats (top-right) exhibit both high shot volume and high accuracy, distinguishing them from "speculative" shooters who rely on high volume to find the net.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Standard_Sh/90', y='Standard_SoT%', hue='pos_')
plt.title('Shooting Efficiency: Volume vs. Accuracy')

Vis 2: Goal Conversion Efficiency

Caption: By comparing total shots to actual goals, we identify clinical over-performers. Players significantly above the regression trendline demonstrate superior finishing skill relative to their shot frequency.

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df, x='Standard_Sh', y='Standard_Gls', scatter_kws={'alpha':0.5})
plt.title('Total Shots vs. Goals Scored')
plt.show()

Vis 3: Distribution of Goals by Primary Position

Caption: This boxplot illustrates the goal-scoring floor and ceiling for different tactical roles. Note the variance in 'MF' (Midfielder), which suggests a mix of holding and attacking roles requiring further segmentation.

In [ ]:
df['Primary_Pos'] = df['pos_'].str.split(',').str[0]

print("Columns in DataFrame:", df.columns.tolist())

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df, 
    x='Primary_Pos', 
    y='Standard_Gls', 
    hue='Primary_Pos', 
    palette='viridis', 
    legend=False
)
plt.title('Distribution of Goals by Primary Position')
plt.ylabel('Goals Scored')
plt.show()

Vis 4: Top 15 Players: Shots on Target per 90

Caption: This bar chart ranks the league's most consistent threats. Focusing on Shots on Target (SoT) rather than total shots provides a cleaner metric of a player’s ability to force goalkeeper intervention.

In [ ]:
plt.figure(figsize=(10, 6))
top_shooters = df.nlargest(15, 'Standard_SoT/90')
sns.barplot(data=top_shooters, x='Standard_SoT/90', y='player', hue='team', dodge=False)
plt.title('Top 15 Players: Shots on Target per 90')
plt.show()

Vis 5: Correlation Heatmap of Shooting Metrics

Caption: The heatmap reveals the strength of relationships between shooting metrics. Strong correlations between volume and output are expected, but weak correlations with SoT% suggest that accuracy may be a distinct, trainable skill set.

In [ ]:
df['Primary_Pos'] = df['pos_'].str.split(',').str[0]

print(df.columns) 

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df, 
    x='Primary_Pos', 
    y='Standard_Sh/90', 
    hue='Primary_Pos', 
    palette='muted',
    legend=False
)
plt.title('Shot Volume by Tactical Role (The "Aggression" Metric)')
plt.show()

Vis 6: Open Play vs. Set Piece Reliance

Caption: This plot differentiates between goal sources. Players falling below the identity line rely heavily on penalties; those on or above the line are consistent contributors during active, open-play sequences.

In [ ]:
df['Primary_Pos'] = df['pos_'].str.split(',').str[0]
df['Non_Penalty_Goals'] = df['Standard_Gls'] - df['Standard_PK']

print("Columns present:", df.columns.tolist())

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='Standard_Gls', 
    y='Non_Penalty_Goals', 
    hue='Primary_Pos', 
    s=100
)

max_val = df['Standard_Gls'].max()
plt.plot([0, max_val], [0, max_val], 'r--', alpha=0.7, label='100% Open Play')

plt.title('Open Play vs. Set Piece Reliance')
plt.xlabel('Total Goals')
plt.ylabel('Non-Penalty Goals')
plt.legend()
plt.show()

Summary of Initial Findings
Our exploratory analysis of the 2025-2026 Premier League dataset reveals several critical patterns that challenge traditional positional nomenclature:

Positional Blur: A significant number of players listed as "MF" (Midfielders) exhibit shooting volume and goal-contribution profiles identical to "FW" (Forwards). This confirms that squad-list positions are insufficient for modern tactical analysis, validating our "Role-DNA" approach.

Performance Archetypes: We identified a clear distinction between volume-based threats (who shoot frequently but with lower precision) and clinical finishers (who possess high SoT% and goal-conversion ratios).

Metric Interdependence: Our heatmap analysis highlights that while total shots are highly correlated with total goals, the Standard_SoT% metric functions as a unique indicator of "finishing quality" rather than just offensive involvement.

Set-Piece vs. Open-Play: Plotting non-penalty goals against total goals provided a clear "Reliability Index." Players clustering below the identity line suggest a tactical reliance on set-piece situations, which may distinguish them from "Active-Play Creators."

Hypotheses for Further Exploration
Building on these initial EDA insights, the next phase of this project will test the following hypotheses through clustering and predictive modeling:

Hypothesis A: Role Convergence Across Teams. We hypothesize that by applying K-Nearest Neighbors (KNN) to a multidimensional feature space (including progressive actions, touch density, and shot-creating actions), we will find that "Inside Forwards" from different clubs occupy the same statistical cluster, regardless of their disparate listed positions or team systems.

Hypothesis B: Predictive Power of "Role-DNA." We anticipate that our "Role-DNA" dashboard—which benchmarks players against their tactical peers rather than positional peers—will provide a higher correlation with future goal-scoring consistency than traditional metrics like age or current market-listed position.

Hypothesis C: Spatial Impact on Tactical Utility. We hypothesize that players demonstrating high shot-creating actions (SCA) combined with specific defensive-third touch-density will statistically emerge as "Deep-Lying Playmakers," a role currently obscured by the broad label of "Midfielder" in standard datasets.